# CV Production Quantization Lab
This notebook extends the production lab into a realistic computer vision workflow. It demonstrates dataset preparation, model loading, post-training quantization, ONNX export, benchmarking, and a simple prediction application suitable for free Colab GPU.

## 1. Colab GPU Setup and Dependencies
Install required packages, verify the GPU runtime, and configure the device for training and inference.

In [ ]:
# Install required packages for Colab
!pip -q install timm onnx onnxruntime onnxruntime-tools onnxsim torch torchvision datasets

import os
import time
import torch
import timm
import onnx
import onnxruntime as ort
from pathlib import Path

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("ONNX Runtime providers:", ort.get_available_providers())

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

WORKDIR = Path("/content/cv_quant_prod")
WORKDIR.mkdir(exist_ok=True)
os.chdir(WORKDIR)
print("Workspace:", WORKDIR)

## 2. Dataset Loading and Preprocessing
Load a sample CV dataset, preprocess images, create data loaders, and visualize examples for a realistic production scenario.

In [ ]:
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import tarfile
from pathlib import Path

BATCH_SIZE = 64
IMAGE_SIZE = 224

transform_train = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transform_val = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# --- Fast CIFAR-10 download ---
# Try Hugging Face Hub first: it's served off a real CDN (Cloudflare),
# which is typically much faster and more reliable than the single
# academic server torchvision downloads from by default. If that fails
# for any reason (package issue, repo unavailable, network policy), we
# fall back to the verified tarball approach from before: aria2c
# (parallel) -> wget (single connection) -> torchvision's own downloader,
# with integrity checked by actually extracting and confirming the
# expected files are present.

class HFCIFAR10Dataset(Dataset):
    """Wraps a Hugging Face CIFAR-10 split to look like torchvision's CIFAR10."""
    def __init__(self, hf_split, transform=None):
        self.data = hf_split
        self.transform = transform
        self.classes = hf_split.features["label"].names

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        img = item["img"].convert("RGB")
        label = item["label"]
        if self.transform:
            img = self.transform(img)
        return img, label

train_dataset = None
val_dataset = None

try:
    print("Attempt 0: downloading CIFAR-10 from Hugging Face Hub (CDN-backed)...")
    from datasets import load_dataset
    hf_cifar = load_dataset("uoft-cs/cifar10")
    train_dataset = HFCIFAR10Dataset(hf_cifar["train"], transform=transform_train)
    val_dataset = HFCIFAR10Dataset(hf_cifar["test"], transform=transform_val)
    print("Loaded CIFAR-10 from Hugging Face Hub.")
except Exception as e:
    print(f"Hugging Face download failed ({e.__class__.__name__}: {e}); falling back to tarball download.")

if train_dataset is None or val_dataset is None:
    DATA_DIR = Path("./data")
    DATA_DIR.mkdir(exist_ok=True)
    CIFAR_TAR = DATA_DIR / "cifar-10-python.tar.gz"
    CIFAR_EXTRACTED = DATA_DIR / "cifar-10-batches-py"
    OFFICIAL_URL = "https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz"
    EXPECTED_FILES = [
        "data_batch_1", "data_batch_2", "data_batch_3", "data_batch_4", "data_batch_5",
        "test_batch", "batches.meta",
    ]

    def extraction_is_complete(extracted_dir):
        return extracted_dir.exists() and all((extracted_dir / f).exists() for f in EXPECTED_FILES)

    def try_extract(tar_path, dest_dir):
        """Attempt extraction; return True on success, False on any corruption error."""
        try:
            with tarfile.open(tar_path) as tar:
                tar.extractall(dest_dir, filter="data")
            return extraction_is_complete(CIFAR_EXTRACTED)
        except (tarfile.TarError, EOFError, OSError) as e:
            print(f"Extraction failed ({e.__class__.__name__}: {e}) -- archive is likely truncated/corrupt.")
            return False

    if not extraction_is_complete(CIFAR_EXTRACTED):
        success = False

        print("Attempt 1: aria2c parallel download from the official source...")
        if CIFAR_TAR.exists():
            CIFAR_TAR.unlink()
        !apt-get -qq install -y aria2 > /dev/null 2>&1
        !aria2c -x 16 -s 16 -k 1M -d "{DATA_DIR}" -o cifar-10-python.tar.gz "{OFFICIAL_URL}"
        if CIFAR_TAR.exists():
            success = try_extract(CIFAR_TAR, DATA_DIR)

        if not success:
            print("Attempt 2: falling back to a single-connection wget...")
            if CIFAR_TAR.exists():
                CIFAR_TAR.unlink()
            !wget -q -O "{CIFAR_TAR}" "{OFFICIAL_URL}"
            if CIFAR_TAR.exists():
                success = try_extract(CIFAR_TAR, DATA_DIR)

        if not success:
            print("Attempt 3: falling back to torchvision's built-in downloader...")
            if CIFAR_TAR.exists():
                CIFAR_TAR.unlink()
            from torchvision.datasets.utils import download_and_extract_archive
            download_and_extract_archive(OFFICIAL_URL, download_root=str(DATA_DIR))
            success = extraction_is_complete(CIFAR_EXTRACTED)

        assert success, "All download methods failed to produce a valid CIFAR-10 dataset."
        print("CIFAR-10 downloaded and verified.")
    else:
        print("CIFAR-10 already present locally, skipping download.")

    train_dataset = torchvision.datasets.CIFAR10(root=str(DATA_DIR), train=True, download=False, transform=transform_train)
    val_dataset = torchvision.datasets.CIFAR10(root=str(DATA_DIR), train=False, download=False, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

# A small, fixed random subset of the validation set for fast accuracy checks.
# 500 images (~50 per class) is plenty to see whether quantization degrades
# accuracy -- running the full 10,000-image val set on every quantization
# variant just burns Colab time without changing the conclusion. Increase
# EVAL_SUBSET_SIZE if you want a tighter estimate for a final report.
EVAL_SUBSET_SIZE = 500
eval_indices = torch.randperm(len(val_dataset), generator=torch.Generator().manual_seed(42))[:EVAL_SUBSET_SIZE]
eval_subset = torch.utils.data.Subset(val_dataset, eval_indices.tolist())
eval_loader = DataLoader(eval_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

classes = train_dataset.classes
print("Classes:", classes)
print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print(f"Fast-eval subset: {len(eval_subset)} samples")

images, labels = next(iter(train_loader))
fig, axs = plt.subplots(2, 5, figsize=(12, 5))
for i in range(10):
    ax = axs[i // 5, i % 5]
    img = images[i].cpu().permute(1, 2, 0).numpy()
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]
    img = img.clip(0, 1)
    ax.imshow(img)
    ax.set_title(classes[labels[i]])
    ax.axis("off")
plt.tight_layout()

## 3. CV Model Loading and Transfer Learning
Define or load a common CV model architecture, then evaluate it in a transfer learning scenario.

In [ ]:
model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=10)
model.to(DEVICE)
model.eval()

params = sum(p.numel() for p in model.parameters())
print(f"Loaded model with {params / 1e6:.2f}M parameters")

@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    total, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
    return correct / total

sample_accuracy = evaluate(model, eval_loader, DEVICE)
print(f"Before fine-tuning (untrained head, expect ~chance level) accuracy on {len(eval_subset)}-sample subset: {sample_accuracy:.4f}")

## 3b. Fine-Tuning the Classifier Head
Freeze the pretrained backbone and train only the new 10-class head so accuracy numbers are meaningful, before moving on to quantization. Uses a small training subset and a handful of epochs to stay fast on a free Colab T4.

In [ ]:
import torch.nn as nn

# Freeze the backbone; only the classifier head will be trained. This keeps
# fine-tuning fast (few trainable params) and is standard practice for
# transfer learning when the backbone was pretrained on a large dataset.
for param in model.parameters():
    param.requires_grad = False
classifier = model.get_classifier()
for param in classifier.parameters():
    param.requires_grad = True

model.to(DEVICE)

# Small, fixed-seed training subset -- like the eval subset, this is about
# getting a real (non-random) accuracy signal quickly, not squeezing out
# maximum accuracy. Bump TRAIN_SUBSET_SIZE / NUM_EPOCHS for a stronger fit.
TRAIN_SUBSET_SIZE = 4000
NUM_EPOCHS = 3
LEARNING_RATE = 1e-3

train_indices = torch.randperm(len(train_dataset), generator=torch.Generator().manual_seed(42))[:TRAIN_SUBSET_SIZE]
finetune_subset = torch.utils.data.Subset(train_dataset, train_indices.tolist())
finetune_loader = DataLoader(finetune_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

optimizer = torch.optim.Adam(classifier.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(NUM_EPOCHS):
    running_loss, running_correct, running_total = 0.0, 0, 0
    for images, labels in finetune_loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        running_correct += (outputs.argmax(dim=1) == labels).sum().item()
        running_total += labels.size(0)

    epoch_loss = running_loss / running_total
    epoch_acc = running_correct / running_total
    print(f"Epoch {epoch + 1}/{NUM_EPOCHS} - loss: {epoch_loss:.4f} - train accuracy: {epoch_acc:.4f}")

model.eval()
finetuned_accuracy = evaluate(model, eval_loader, DEVICE)
print(f"After fine-tuning accuracy on {len(eval_subset)}-sample subset: {finetuned_accuracy:.4f}")

## 4. Post-Training Quantization Workflow
Apply dynamic quantization and compare the quantized model to the original FP32 model.

In [ ]:
import copy
from torch.quantization import quantize_dynamic

model_fp32 = copy.deepcopy(model).to("cpu")
model_dynamic = quantize_dynamic(model_fp32, {torch.nn.Linear}, dtype=torch.qint8)

print("Dynamic quantized model created.")

@torch.no_grad()
def evaluate_cpu(model, loader):
    model.eval()
    total, correct = 0, 0
    for images, labels in loader:
        images, labels = images.to("cpu"), labels.to("cpu")
        outputs = model(images)
        preds = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (preds == labels).sum().item()
    return correct / total

quant_accuracy = evaluate_cpu(model_dynamic, eval_loader)
print(f"Dynamic quantized CPU validation accuracy (on {len(eval_subset)}-sample subset): {quant_accuracy:.4f}")

## 5. Export and Save Quantized Model
Export the FP32 model to ONNX, then use ONNX Runtime's own quantization tool to produce an INT8 ONNX model. (PyTorch's dynamically-quantized ops don't have a supported ONNX export path -- quantizing the ONNX graph directly, after export, is the standard way to get a portable quantized model.) Save artifacts and verify graph validity for both.

In [ ]:
fp32_onnx_path = WORKDIR / "efficientnet_b0_fp32.onnx"
quant_onnx_path = WORKDIR / "efficientnet_b0_dynamic.onnx"
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, dtype=torch.float32)

# Export the FP32 model (not the PyTorch-quantized one -- quantized::linear_dynamic
# has no supported ONNX opset mapping and will raise UnsupportedOperatorError).
# dynamo=False: recent PyTorch defaults to the newer "dynamo" exporter, which
# needs the extra onnxscript package; the legacy exporter needs no extra deps
# and is fine here since we're exporting a plain FP32 graph.
torch.onnx.export(
    model_fp32,
    dummy_input,
    fp32_onnx_path,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    opset_version=18,
    dynamo=False,
)

print(f"Exported FP32 ONNX model to: {fp32_onnx_path}")
print(f"FP32 ONNX file size: {fp32_onnx_path.stat().st_size / 1e6:.2f} MB")

fp32_onnx_model = onnx.load(fp32_onnx_path)
onnx.checker.check_model(fp32_onnx_model)
print("FP32 ONNX model validation passed.")

# Now quantize the ONNX graph itself (INT8 weights) using ONNX Runtime's own
# quantization tool -- this produces standard QLinearMatMul/DynamicQuantizeLinear
# ops that any ONNX Runtime can execute, unlike PyTorch's native quantized ops.
from onnxruntime.quantization import quantize_dynamic as onnx_quantize_dynamic, QuantType

onnx_quantize_dynamic(
    model_input=str(fp32_onnx_path),
    model_output=str(quant_onnx_path),
    weight_type=QuantType.QInt8,
)

print(f"Exported dynamically quantized ONNX model to: {quant_onnx_path}")
print(f"Quantized ONNX file size: {quant_onnx_path.stat().st_size / 1e6:.2f} MB")

quant_onnx_model = onnx.load(quant_onnx_path)
onnx.checker.check_model(quant_onnx_model)
print("Quantized ONNX model validation passed.")

## 6. Production Inference Demo
Run inference with the quantized ONNX model, measure latency, and compare outputs to the original model.

In [ ]:
import numpy as np

providers = ["CPUExecutionProvider"]
ort_session = ort.InferenceSession(str(quant_onnx_path), providers=providers)
print("ORT providers used:", ort_session.get_providers())

def run_onnx(session, image):
    inputs = {session.get_inputs()[0].name: image.astype(np.float32)}
    return session.run(None, inputs)[0]

images, labels = next(iter(val_loader))
images_np = images[:16].numpy()

for _ in range(5):
    _ = run_onnx(ort_session, images_np)

start = time.perf_counter()
for _ in range(20):
    _ = run_onnx(ort_session, images_np)
end = time.perf_counter()
print(f"Average ONNX inference time for batch 16: {(end - start) / 20 * 1000:.2f} ms")

onnx_preds = run_onnx(ort_session, images_np[:8])
onnx_labels = np.argmax(onnx_preds, axis=1)
print("Sample ONNX predictions:", onnx_labels)
print("Ground truth:", labels[:8].numpy())

## 7. Simple Prediction Application Integration
Build a lightweight prediction utility that loads the quantized model and returns class predictions for new images.

In [ ]:
from PIL import Image

def preprocess_image(image: Image.Image) -> np.ndarray:
    transform = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    tensor = transform(image).unsqueeze(0)
    return tensor.numpy().astype(np.float32)

def predict_image_onnx(image_path: str, session: ort.InferenceSession):
    image = Image.open(image_path).convert("RGB")
    input_data = preprocess_image(image)
    outputs = session.run(None, {session.get_inputs()[0].name: input_data})[0]
    prediction = int(np.argmax(outputs, axis=1)[0])
    return classes[prediction], float(np.max(outputs))

sample_image_path = WORKDIR / "sample_image.png"
sample_pil = torchvision.transforms.ToPILImage()(images[0].cpu())
sample_pil.save(sample_image_path)

label, confidence = predict_image_onnx(str(sample_image_path), ort_session)
print(f"Predicted class: {label}, confidence: {confidence:.4f}")

def serve_prediction(image_path: str):
    session = ort.InferenceSession(str(quant_onnx_path), providers=["CPUExecutionProvider"])
    label, confidence = predict_image_onnx(image_path, session)
    return {
        "predicted_label": label,
        "confidence": confidence,
        "model": str(quant_onnx_path.name),
    }

print(serve_prediction(str(sample_image_path)))